In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential
from tensorflow.keras import Input
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.activations import sigmoid

Code to download training set

In [ ]:
import os
import json
from getpass import getpass

# Enter Kaggle credentials securely
KAGGLE_USERNAME = input("Enter your Kaggle username: ")
KAGGLE_KEY = getpass("Enter your Kaggle API token: ")

# Create Kaggle config directory
os.makedirs("/root/.kaggle", exist_ok=True)

# Save credentials to kaggle.json
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump({
        "username": "sonusurendran",
        "key": "KGAT_a153b4de8a11ce2f119dc08d7d717821"
    }, f)

# Set permission
!chmod 600 /root/.kaggle/kaggle.json

Enter your Kaggle username: sonusurendran
Enter your Kaggle API token: ··········


In [ ]:
import os

DATASET_NAME = "olafkrastovski/handwritten-digits-0-9"
SAVE_DIR = "/content/image"

os.makedirs(SAVE_DIR, exist_ok=True)

# Download and unzip dataset into /content/image
!kaggle datasets download -d {DATASET_NAME} -p {SAVE_DIR} --unzip

print("Dataset downloaded and saved to:", SAVE_DIR)

Dataset URL: https://www.kaggle.com/datasets/olafkrastovski/handwritten-digits-0-9
License(s): CC0-1.0
100% 66.7M/66.7M [00:04<00:00, 16.2MB/s]

Dataset downloaded and saved to: /content/image


Using numpy to load the data

In [ ]:
import os
import numpy as np
from PIL import Image

DATA_DIR = "/content/image"
IMG_SIZE = (28, 28)

selected_classes = ["0", "1"]

X = []
y = []

for class_name in selected_classes:
    class_folder = os.path.join(DATA_DIR, class_name)

    if not os.path.exists(class_folder):
        print("Folder not found:", class_folder)
        continue

    for file_name in os.listdir(class_folder):
        file_path = os.path.join(class_folder, file_name)

        try:
            img = Image.open(file_path).convert("L")
            img = img.resize(IMG_SIZE)

            img_array = np.array(img)

            X.append(img_array)
            y.append(int(class_name))

        except Exception as e:
            print("Skipped:", file_path, e)

X = np.array(X)
y = np.array(y)

# Normalize pixel values
X = X / 255.0

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Labels:", np.unique(y))

X shape: (4477, 28, 28)
y shape: (4477,)
Labels: [0 1]


In [ ]:
x_train = X.reshape(X.shape[0], 28 * 28)

In [ ]:
x_train.shape

(4477, 784)

Now building a binary classification model to predict 0 or 1

In [ ]:
model = Sequential ([
    Input(shape=(784,)),
    Dense(units=25, activation='relu', name="layer1"),
    Dense(units=15, activation='relu', name="layer2"),
    Dense(units=1, activation=sigmoid, name="output")
])

In [ ]:
model.compile(
    loss=BinaryCrossentropy(),
    optimizer= tf.keras.optimizers.Adam(learning_rate=0.01)
)

In [ ]:
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ layer1 (Dense)                  │ (None, 25)             │        19,625 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer2 (Dense)                  │ (None, 15)             │           390 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │            16 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,031 (78.25 KB)

 Trainable params: 20,031 (78.25 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.fit(x_train, y, epochs=10, batch_size=32)

Epoch 1/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.6704
Epoch 2/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.3303
Epoch 3/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.3165
Epoch 4/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2475
Epoch 5/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2506
Epoch 6/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2996
Epoch 7/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2197
Epoch 8/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2366
Epoch 9/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2355
Epoch 10/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2698


In [ ]:
type(model.get_layer("layer1").get_weights())

list

In [ ]:
pred = model.predict(x_train)

140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


In [ ]:
pred = (pred >= 0.5).astype(int)

In [ ]:
pred

array([[0],
       [0],
       [0],
       ...,
       [1],
       [0],
       [1]])

In [ ]:
m = len(pred)
mismatch = 0

for i in range(m):
  if pred[i] != y[i]:
    mismatch += 1
accuracy = m - mismatch
precentage = (accuracy/m) * 100
print(f"accuracy is : {precentage}")

accuracy is : 93.38842975206612
